In [1]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import os
from tqdm import tqdm
from PIL import Image
import numpy as np

In [2]:
# Paths
processed_dir = r"C:\Users\Legion\Desktop\pneumo\data\processed"
balanced_dir = r"C:\Users\Legion\Desktop\pneumo\data\balanced"

In [3]:
# Step 1: Load all data into a single dataset
def load_images_and_labels(directory, img_size=(224, 224)):
    images, labels = [], []
    for label in os.listdir(directory):
        class_dir = os.path.join(directory, label)
        for img_file in tqdm(os.listdir(class_dir), desc=f"Loading {label}"):
            img_path = os.path.join(class_dir, img_file)
            try:
                img = Image.open(img_path).resize(img_size)
                images.append(np.array(img))
                labels.append(label)
            except Exception as e:
                print(f"Error loading image {img_file}: {e}")
    return np.array(images), np.array(labels)

train_images, train_labels = load_images_and_labels(os.path.join(processed_dir, "train"))
val_images, val_labels = load_images_and_labels(os.path.join(processed_dir, "val"))
test_images, test_labels = load_images_and_labels(os.path.join(processed_dir, "test"))

# Combine all data
all_images = np.concatenate([train_images, val_images, test_images], axis=0)
all_labels = np.concatenate([train_labels, val_labels, test_labels], axis=0)

# Step 2: Flatten images for SMOTE compatibility
all_images_flat = all_images.reshape(all_images.shape[0], -1)

# Convert labels to numeric using LabelEncoder
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
all_labels_encoded = label_encoder.fit_transform(all_labels)

# Apply SMOTE
smote = SMOTE(sampling_strategy="minority", random_state=42)
balanced_images_flat, balanced_labels_encoded = smote.fit_resample(all_images_flat, all_labels_encoded)

# Restore balanced images to original shape
balanced_images = balanced_images_flat.reshape(-1, all_images.shape[1], all_images.shape[2], all_images.shape[3])
balanced_labels = label_encoder.inverse_transform(balanced_labels_encoded)

# Step 3: Re-split the balanced dataset
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

X_train, X_temp, y_train, y_temp = train_test_split(balanced_images, balanced_labels, test_size=(1 - train_ratio), random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=(test_ratio / (test_ratio + val_ratio)), random_state=42)

# Step 4: Save the balanced and split datasets
def save_balanced_data(images, labels, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for label in np.unique(labels):
        label_dir = os.path.join(output_dir, label)
        os.makedirs(label_dir, exist_ok=True)
        for i, img in enumerate(images[labels == label]):
            save_path = os.path.join(label_dir, f"image_{i}.png")
            Image.fromarray(img).save(save_path)

save_balanced_data(X_train, y_train, os.path.join(balanced_dir, "train"))
save_balanced_data(X_val, y_val, os.path.join(balanced_dir, "val"))
save_balanced_data(X_test, y_test, os.path.join(balanced_dir, "test"))

print(f"Balanced dataset saved at {balanced_dir}")

Loading PNEUMONIA: 100%|██████████| 390/390 [00:00<00:00, 1595.86it/s]


Balanced dataset saved at C:\Users\Legion\Desktop\pneumo\data\balanced
